# TUGAS MINGGU 10 — IMPLEMENTASI ETL PIPELINE
**Mata Kuliah:** Big Data  
**Kelompok 8**
- Alif Akbar Arraiyan — 101032330121
- Farid Yusron Syabani — 101032300192

## Ringkasan
Notebook ini mengimplementasikan pipeline **ETL (Extract – Transform – Load)** untuk studi kasus
**Analisis Kecelakaan Lalu Lintas Amerika Serikat (2020–2024)**, melanjutkan rancangan arsitektur
Minggu 9: *Apache Beam (DirectRunner) → Supabase PostgreSQL (skema `analytics`) → Power BI*,
dengan **Apache Airflow** sebagai orkestrator.

Implementasi di notebook ini dijalankan secara lokal dengan **pandas** sebagai pengganti Apache Beam
(model alur dan tahapannya sama), dan **SQLite** sebagai representasi lokal dari tabel
`analytics.accidents` di Supabase. Jika variabel lingkungan `SUPABASE_DB_URL` tersedia, fungsi
`load_to_supabase()` di bawah akan menulis langsung ke Supabase PostgreSQL melalui SQLAlchemy.

**Sumber data:** `data/processed/US_Accidents_Cleaned_NHTSA_2020-2024.csv` (hasil gathering Minggu 3)
**Referensi eksternal:** NHTSA CrashAPI — `definitions/GetStates` (disimpan lokal karena endpoint
live mengembalikan HTTP 403 dari lingkungan ini).

Urutan eksekusi: **Extract → Transform → Load → Integrasi → Testing**.

In [1]:
import os, sys, json, time, sqlite3, re
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np


def find_project_root(start=Path.cwd()):
    p = Path(start).resolve()
    for cand in [p] + list(p.parents):
        if (cand / "data" / "processed").is_dir():
            return cand
    raise FileNotFoundError("Root repository tidak ditemukan (folder data/processed tidak ada)")


PROJECT_ROOT = find_project_root()
SOURCE_CSV   = PROJECT_ROOT / "data" / "processed" / "US_Accidents_Cleaned_NHTSA_2020-2024.csv"
API_SNAPSHOT = PROJECT_ROOT / "data" / "raw" / "nhtsa_api_live.json"
API_ENDPOINT = ("https://crashviewer.nhtsa.dot.gov/crashviewer/CrashAPI/crashes/GetCaseList"
                "?states=1,51&fromYear={year}&toYear={year}"
                "&minNumOfVehicles=1&maxNumOfVehicles=6&format=json")
DELIVERABLE  = PROJECT_ROOT / "tugas" / "minggu10" / "Tugas_Minggu10_Kelompok8"
for sub in ["Source_Code", "Dataset", "Documentation", "Screenshots"]:
    (DELIVERABLE / sub).mkdir(parents=True, exist_ok=True)

TRANSFORMED_CSV = DELIVERABLE / "Dataset" / "transformed_dataset.csv"
API_CASES_CSV   = DELIVERABLE / "Dataset" / "nhtsa_api_cases.csv"
WAREHOUSE_DB    = DELIVERABLE / "Dataset" / "supabase_local.db"
METRICS_JSON    = DELIVERABLE / "Documentation" / "etl_metrics.json"
LOG_FILE        = DELIVERABLE / "Screenshots" / "pipeline_log.txt"

LOGS = []

def log(msg, level="INFO"):
    line = f"[{datetime.now():%H:%M:%S}] [{level}] {msg}"
    LOGS.append(line)
    print(line, flush=True)

NHTSA_STATES = {
    "AK": (2, "Alaska", "West"),
    "AL": (1, "Alabama", "South"),
    "AR": (5, "Arkansas", "South"),
    "AZ": (4, "Arizona", "West"),
    "CA": (6, "California", "West"),
    "CO": (8, "Colorado", "West"),
    "CT": (9, "Connecticut", "Northeast"),
    "DC": (11, "District of Columbia", "South"),
    "DE": (10, "Delaware", "South"),
    "FL": (12, "Florida", "South"),
    "GA": (13, "Georgia", "South"),
    "HI": (15, "Hawaii", "West"),
    "IA": (19, "Iowa", "Midwest"),
    "ID": (16, "Idaho", "West"),
    "IL": (17, "Illinois", "Midwest"),
    "IN": (18, "Indiana", "Midwest"),
    "KS": (20, "Kansas", "Midwest"),
    "KY": (21, "Kentucky", "South"),
    "LA": (22, "Louisiana", "South"),
    "MA": (25, "Massachusetts", "Northeast"),
    "MD": (24, "Maryland", "South"),
    "ME": (23, "Maine", "Northeast"),
    "MI": (26, "Michigan", "Midwest"),
    "MN": (27, "Minnesota", "Midwest"),
    "MO": (29, "Missouri", "Midwest"),
    "MS": (28, "Mississippi", "South"),
    "MT": (30, "Montana", "West"),
    "NC": (37, "North Carolina", "South"),
    "ND": (38, "North Dakota", "Midwest"),
    "NE": (31, "Nebraska", "Midwest"),
    "NH": (33, "New Hampshire", "Northeast"),
    "NJ": (34, "New Jersey", "Northeast"),
    "NM": (35, "New Mexico", "West"),
    "NV": (32, "Nevada", "West"),
    "NY": (36, "New York", "Northeast"),
    "OH": (39, "Ohio", "Midwest"),
    "OK": (40, "Oklahoma", "South"),
    "OR": (41, "Oregon", "West"),
    "PA": (42, "Pennsylvania", "Northeast"),
    "PR": (43, "Puerto Rico", "South"),
    "RI": (44, "Rhode Island", "Northeast"),
    "SC": (45, "South Carolina", "South"),
    "SD": (46, "South Dakota", "Midwest"),
    "TN": (47, "Tennessee", "South"),
    "TX": (48, "Texas", "South"),
    "UT": (49, "Utah", "West"),
    "VA": (51, "Virginia", "South"),
    "VI": (52, "Virgin Islands", "South"),
    "VT": (50, "Vermont", "Northeast"),
    "WA": (53, "Washington", "West"),
    "WI": (55, "Wisconsin", "Midwest"),
    "WV": (54, "West Virginia", "South"),
    "WY": (56, "Wyoming", "West")
}

def build_reference():
    """Tabel referensi state dari NHTSA CrashAPI (versi cached)."""
    return pd.DataFrame(
        [(k, v[0], v[1], v[2]) for k, v in NHTSA_STATES.items()],
        columns=["State", "NHTSA_State_ID_Ref", "NHTSA_State_Name_Ref", "NHTSA_Region_Ref"],
    )

log("Konfigurasi siap | root: " + str(PROJECT_ROOT))

[09:01:13] [INFO] Konfigurasi siap | root: C:\Punya Alif\bigdata


## 1. Extract — Data Extraction

**Sumber data**
1. Dataset utama (Kaggle, `US_Accidents_March23.csv`) yang telah dikonsolidasi pada Minggu 3 menjadi
   `US_Accidents_Cleaned_NHTSA_2020-2024.csv` — 266.656 baris, 21 kolom, mencakup 2020–2024.
2. **Data baru langsung dari NHTSA Crash API** (bukan hasil cleaning Minggu 3):
   `GET /crashes/GetCaseList?states=1,51&fromYear=2020&toYear=2026&minNumOfVehicles=1&maxNumOfVehicles=6&format=json`.
   Endpoint ini menolak request `urllib`/`curl` (HTTP 403), jadi respons diambil lewat browser
   per tahun (2020–2024; 2025–2026 kosong) menghasilkan **8.798 kasus kecelakaan fatal**,
   disimpan sebagai snapshot `data/raw/nhtsa_api_live.json`.
3. Tabel referensi state dari endpoint NHTSA CrashAPI `definitions/GetStates` (53 entitas:
   ID, kode, nama lengkap, region). Endpoint live juga gagal via `urllib`, sehingga respons
   tersimpan (`cached`) yang dipakai; ini tidak mengubah skema maupun isi referensi.

**Metode:** `pandas.read_csv()` untuk sumber utama, `json.load()` untuk snapshot API, dan
dictionary referensi untuk sumber ketiga. Semuanya ditaruh di RAM sebagai DataFrame sebelum
masuk ke tahap Transform — sesuai rancangan "penarikan langsung ke RAM, tanpa staging area"
pada arsitektur Minggu 9.

In [2]:
def extract(path):
    log("EXTRACT  | Memulai pengambilan data dari sumber ...")
    t0 = time.time()
    df = pd.read_csv(path, low_memory=False)
    ref = build_reference()
    log(f"EXTRACT  | Dataset utama: {len(df):,} baris x {df.shape[1]} kolom")
    log(f"EXTRACT  | Referensi NHTSA CrashAPI: {len(ref)} state berhasil dimuat")
    log(f"EXTRACT  | Selesai dalam {time.time() - t0:.2f} detik")
    info = {
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1]),
        "columns": list(df.columns),
        "dtypes": {c: str(t) for c, t in df.dtypes.items()},
        "nulls": int(df.isna().sum().sum()),
        "duplicates": int(df.duplicated().sum()),
        "duplicate_id": int(df["ID"].duplicated().sum()),
        "weather_categories": int(df["Weather_Condition"].nunique()),
        "sample": df.head(5).where(pd.notna(df.head(5)), None).to_dict("records"),
    }
    return df, ref, info


def fetch_live_api(year):
    """Mencoba mengambil data langsung dari NHTSA Crash API (cadangan bila akses tidak diblokir)."""
    import urllib.request
    url = API_ENDPOINT.format(year=year)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
        "Accept": "application/json",
        "Referer": "https://crashviewer.nhtsa.dot.gov/CrashAPI",
    }
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.loads(response.read().decode("utf-8", errors="replace"))


def extract_api_snapshot(path):
    """Membaca snapshot respons API NHTSA yang diambil lewat browser per tahun."""
    log("EXTRACT  | Membaca snapshot data baru dari NHTSA Crash API ...")
    t0 = time.time()
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows = [item for group in payload.get("Results", [])
            for item in (group if isinstance(group, list) else [group])]
    df = pd.DataFrame(rows)
    info = {
        "rows": int(len(df)),
        "cols": int(df.shape[1]),
        "count_reported": int(payload.get("Count", len(df))),
        "columns": list(df.columns),
        "endpoint": API_ENDPOINT.format(year="2020..2026"),
        "sample": df.head(5).to_dict("records"),
    }
    log(f"EXTRACT  | NHTSA Crash API: {len(df):,} kasus baru | {len(df.columns)} atribut")
    log(f"EXTRACT  | Endpoint: .../GetCaseList?states=1,51&fromYear=2020&toYear=2026&format=json")
    log(f"EXTRACT  | Snapshot API dibaca dalam {time.time() - t0:.2f} detik")
    return df, info


df_raw, df_ref, extract_info = extract(SOURCE_CSV)
df_api_raw, api_info = extract_api_snapshot(API_SNAPSHOT)
log("EXTRACT  | Pemeriksaan awal dataset utama:")
log(f"EXTRACT  |   missing values  : {extract_info['nulls']}")
log(f"EXTRACT  |   baris duplikat  : {extract_info['duplicates']} (duplikat ID: {extract_info['duplicate_id']})")
log(f"EXTRACT  |   tipe Start_Time : {extract_info['dtypes']['Start_Time']} -> perlu dikonversi ke datetime")
log(f"EXTRACT  |   kategori Weather_Condition: {extract_info['weather_categories']} (terlalu banyak untuk BI)")

preview = pd.DataFrame({
    "Kolom": extract_info["columns"],
    "Tipe_data": [extract_info["dtypes"][c] for c in extract_info["columns"]],
    "Missing": [int(df_raw[c].isna().sum()) for c in extract_info["columns"]],
})
display(preview.head(10))
display(df_raw.head(5))
log("EXTRACT  | Contoh 5 baris data baru dari NHTSA Crash API:")
display(df_api_raw.head(5))

[09:01:13] [INFO] EXTRACT  | Memulai pengambilan data dari sumber ...


[09:01:14] [INFO] EXTRACT  | Dataset utama: 266,656 baris x 21 kolom


[09:01:14] [INFO] EXTRACT  | Referensi NHTSA CrashAPI: 53 state berhasil dimuat


[09:01:14] [INFO] EXTRACT  | Selesai dalam 0.74 detik


[09:01:15] [INFO] EXTRACT  | Membaca snapshot data baru dari NHTSA Crash API ...


[09:01:15] [INFO] EXTRACT  | NHTSA Crash API: 8,798 kasus baru | 9 atribut


... (7 outputs left out)

,Kolom,Tipe_data,Missing
0,ID,str,0
1,Severity,int64,0
2,Severity_Label,str,0
3,Start_Time,str,0
4,End_Time,str,0
5,Duration_Minutes,float64,0
6,Hour,int64,0
7,Day_of_Week,str,0
8,City,str,0
9,County,str,0


,ID,Severity,Severity_Label,Start_Time,End_Time,Duration_Minutes,Hour,Day_of_Week,City,County,...,NHTSA_State_ID,NHTSA_State_Name,NHTSA_Region,NHTSA_Fatality_Risk_Tier,NHTSA_API_Latest_FARS_Year,Temperature(F),Humidity(%),Visibility(mi),Weather_Condition,Data_Source
0,NHTSA-2024-483866,4,Severe,2024-12-31 23:25:00,2025-01-01 02:25:00,180.0,23,Tuesday,Harris,Harris,...,48,Texas,South,High,2024,62.0,65.0,10.0,Clear,NHTSA_CrashAPI
1,NHTSA-2024-483853,4,Severe,2024-12-31 23:01:00,2025-01-01 02:01:00,180.0,23,Tuesday,Travis,Travis,...,48,Texas,South,High,2024,62.0,65.0,10.0,Clear,NHTSA_CrashAPI
2,NHTSA-2024-483739,4,Severe,2024-12-31 21:51:00,2025-01-01 00:51:00,180.0,21,Tuesday,Tarrant,Tarrant,...,48,Texas,South,High,2024,62.0,65.0,10.0,Clear,NHTSA_CrashAPI
3,NHTSA-2024-483871,4,Severe,2024-12-31 15:30:00,2024-12-31 18:30:00,180.0,15,Tuesday,Harris,Harris,...,48,Texas,South,High,2024,62.0,65.0,10.0,Clear,NHTSA_CrashAPI
4,NHTSA-2024-483790,4,Severe,2024-12-31 03:00:00,2024-12-31 06:00:00,180.0,3,Tuesday,Cameron,Cameron,...,48,Texas,South,High,2024,62.0,65.0,10.0,Clear,NHTSA_CrashAPI


[09:01:15] [INFO] EXTRACT  | Contoh 5 baris data baru dari NHTSA Crash API:


,CountyName,CrashDate,Fatals,Peds,Persons,St_Case,State,StateName,TotalVehicles
0,ELMORE (51),/Date(1577847480000+0000)/,3,0,4,10001,1,Alabama,1
1,JEFFERSON (73),/Date(1577985480000+0000)/,1,0,6,10002,1,Alabama,4
2,SHELBY (117),/Date(1577976900000+0000)/,1,0,2,10003,1,Alabama,2
3,CALHOUN (15),/Date(1578064800000+0000)/,1,0,5,10004,1,Alabama,1
4,COOSA (37),/Date(1578098700000+0000)/,1,0,1,10005,1,Alabama,1


## 2. Transform — Data Transformation

### Permasalahan yang ditemukan pada data hasil ekstraksi

| No | Permasalahan | Bukti | Dampak | Penanganan |
|----|--------------|-------|--------|------------|
| 1 | `Start_Time` & `End_Time` masih bertipe teks (`str`) | 2 kolom bertipe objek | Tidak bisa dipakai untuk filter/agregasi waktu di Supabase | Konversi ke `datetime64` |
| 2 | `Weather_Condition` memiliki 81 variasi ejaan ("Cloudy", "Mostly Cloudy", "Cloudy / Windy", ...) | 81 unique values | Kardinalitas tinggi, dashboard Power BI menjadi berantakan | Standardisasi ke 7 grup cuaca |
| 3 | Potensi duplikasi berdasarkan `ID` | dicek saat transform | Statistik kecelakaan bisa doble counting | `drop_duplicates(subset='ID')` |
| 4 | Potensi nilai di luar rentang logis (Severity, durasi, suhu, kelembapan, jarak pandang) | dicek saat transform | Outlier merusak analisis | Filter validasi rentang |
| 5 | Atribut NHTSA (`State_ID`, `Nama`, `Region`) berasal dari sumber lama | perlu verifikasi | Bisa tidak sinkron dengan referensi CrashAPI terbaru | Join dengan tabel referensi NHTSA |

Catatan: pada dataset ini tidak ditemukan *missing value* maupun duplikasi aktual. Keduanya tetap
dijalankan sebagai langkah defensif dan dilaporkan hasilnya (nol baris terhapus).

### Teknik yang diterapkan
- **Data Cleaning:** trim/standardisasi teks, konversi tipe tanggal, penghapusan duplikat,
  penanganan nilai tidak valid, validasi referensi.
- **Data Transformation:** *feature creation* (`Accident_Date`, `Month`, `Is_Weekend`,
  `Duration_Category`), *data integration* (join dengan referensi NHTSA),
  *data encoding* (`Weather_Code`), standarisasi kategori cuaca.
- **Data Validation:** assertion akhir untuk missing values, duplikasi, dan tipe data.

In [3]:
def group_weather(value):
    """Mereduksi 81 label cuaca mentah menjadi 7 grup yang stabil untuk analitik."""
    s = str(value).lower().strip()
    if s.startswith("n/a") or s in ("nan", "none", "unknown"):
        return "Other"
    if any(k in s for k in ["thunder", "t-storm", "storm", "squall", "hail"]):
        return "Storm"
    if any(k in s for k in ["snow", "sleet", "freezing", "wintry", "ice"]):
        return "Snow/Ice"
    if any(k in s for k in ["rain", "drizzle", "shower", "precipitation"]):
        return "Rain"
    if any(k in s for k in ["fog", "mist", "haze", "smoke", "dust"]):
        return "Fog/Haze"
    if any(k in s for k in ["cloudy", "overcast"]):
        return "Cloudy"
    if any(k in s for k in ["fair", "clear"]):
        return "Clear"
    return "Other"


def duration_category(minutes):
    if minutes < 30:
        return "Short (<30m)"
    if minutes <= 120:
        return "Medium (30-120m)"
    return "Long (>120m)"


def transform(df, ref):
    log("TRANSFORM| Memulai pembersihan dan transformasi data ...")
    t0 = time.time()
    rows_in, cols_in = df.shape
    issues = {
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_id": int(df["ID"].duplicated().sum()),
        "datetime_string_cols": int(df[["Start_Time", "End_Time"]]
                                    .dtypes.apply(lambda t: t == object).sum()),
        "weather_categories_before": int(df["Weather_Condition"].nunique()),
    }

    clean = df.copy()

    # (a) Standardisasi teks
    text_cols = ["ID", "City", "County", "State", "Weather_Condition",
                 "Severity_Label", "Day_of_Week", "Data_Source"]
    for c in text_cols:
        clean[c] = clean[c].astype(str).str.strip()
    clean["City"] = clean["City"].str.title()
    clean["County"] = clean["County"].str.title()
    clean["State"] = clean["State"].str.upper()
    log(f"TRANSFORM| (a) Standardisasi teks pada {len(text_cols)} kolom")

    # (b) Konversi tipe data tanggal
    clean["Start_Time"] = pd.to_datetime(clean["Start_Time"], errors="coerce")
    clean["End_Time"] = pd.to_datetime(clean["End_Time"], errors="coerce")
    bad_datetime = int(clean[["Start_Time", "End_Time"]].isna().any(axis=1).sum())
    clean = clean.dropna(subset=["Start_Time", "End_Time"]).copy()
    log(f"TRANSFORM| (b) Konversi Start_Time/End_Time ke datetime | baris invalid dihapus: {bad_datetime}")

    # (c) Penghapusan duplikat
    before = len(clean)
    clean = clean.drop_duplicates(subset=["ID"], keep="first").copy()
    removed_dup = before - len(clean)
    log(f"TRANSFORM| (c) Duplicate removal (ID) | baris dihapus: {removed_dup}")

    # (d) Validasi nilai di luar rentang logis
    valid = (
        clean["Severity"].between(1, 4)
        & clean["Duration_Minutes"].between(0, 1440)
        & clean["Temperature(F)"].between(-40, 125)
        & clean["Humidity(%)"].between(0, 100)
        & clean["Visibility(mi)"].between(0, 50)
    )
    removed_invalid = int((~valid).sum())
    clean = clean[valid].copy()
    log(f"TRANSFORM| (d) Invalid value handling | baris dihapus: {removed_invalid}")

    # (e) Data integration: join dengan referensi NHTSA CrashAPI
    merged = clean.merge(ref, on="State", how="left")
    unknown_state = int(merged["NHTSA_State_ID_Ref"].isna().sum())
    mismatch = int((
        (merged["NHTSA_State_ID"] != merged["NHTSA_State_ID_Ref"])
        | (merged["NHTSA_State_Name"] != merged["NHTSA_State_Name_Ref"])
        | (merged["NHTSA_Region"] != merged["NHTSA_Region_Ref"])
    ).sum())
    merged = merged.dropna(subset=["NHTSA_State_ID_Ref"]).copy()
    merged["NHTSA_State_ID"] = merged["NHTSA_State_ID_Ref"].astype(int)
    merged["NHTSA_State_Name"] = merged["NHTSA_State_Name_Ref"]
    merged["NHTSA_Region"] = merged["NHTSA_Region_Ref"]
    merged = merged.drop(columns=["NHTSA_State_ID_Ref", "NHTSA_State_Name_Ref", "NHTSA_Region_Ref"])
    log(f"TRANSFORM| (e) Join referensi NHTSA | state tidak dikenal: {unknown_state} | ketidaksesuaian metadata: {mismatch}")

    # (f) Feature creation
    merged["Accident_Date"] = merged["Start_Time"].dt.date
    merged["Month"] = merged["Start_Time"].dt.month.astype(int)
    merged["Is_Weekend"] = merged["Start_Time"].dt.dayofweek.isin([5, 6]).astype(int)
    merged["Duration_Category"] = merged["Duration_Minutes"].apply(duration_category)

    # (g) Data standardization + encoding
    merged["Weather_Group"] = merged["Weather_Condition"].apply(group_weather)
    categories = sorted(merged["Weather_Group"].unique())
    merged["Weather_Code"] = pd.Categorical(merged["Weather_Group"],
                                            categories=categories).codes.astype(int)
    log(f"TRANSFORM| (f-g) Feature creation & encoding | {issues['weather_categories_before']} -> "
        f"{merged['Weather_Group'].nunique()} grup cuaca")

    # (h) Urutan kolom akhir
    ordered = [
        "ID", "Severity", "Severity_Label", "Start_Time", "End_Time",
        "Accident_Date", "Month", "Hour", "Day_of_Week", "Is_Weekend",
        "Duration_Minutes", "Duration_Category", "City", "County", "State",
        "NHTSA_State_ID", "NHTSA_State_Name", "NHTSA_Region",
        "NHTSA_Fatality_Risk_Tier", "NHTSA_API_Latest_FARS_Year",
        "Temperature(F)", "Humidity(%)", "Visibility(mi)",
        "Weather_Condition", "Weather_Group", "Weather_Code", "Data_Source",
    ]
    merged = merged[ordered].reset_index(drop=True)

    # (i) Validasi akhir
    assert merged.isna().sum().sum() == 0, "Masih ada missing value setelah transformasi"
    assert merged["ID"].duplicated().sum() == 0, "Masih ada duplikat ID setelah transformasi"
    assert str(merged["Start_Time"].dtype).startswith("datetime64"), "Start_Time bukan datetime"
    assert merged["Severity"].between(1, 4).all(), "Severity di luar rentang 1-4"

    report = {
        "rows_in": rows_in, "cols_in": cols_in,
        "rows_out": int(len(merged)), "cols_out": int(merged.shape[1]),
        "removed_bad_datetime": bad_datetime,
        "removed_duplicate": removed_dup,
        "removed_invalid": removed_invalid,
        "unknown_state": unknown_state,
        "metadata_mismatch": mismatch,
        "weather_categories_after": int(merged["Weather_Group"].nunique()),
        "weather_group_counts": {k: int(v) for k, v in merged["Weather_Group"].value_counts().items()},
        "nulls_after": int(merged.isna().sum().sum()),
        "duplicates_after": int(merged["ID"].duplicated().sum()),
        "issues": issues,
        "elapsed_seconds": round(time.time() - t0, 2),
    }
    log(f"TRANSFORM| Selesai: {rows_in:,} -> {len(merged):,} baris | {cols_in} -> {merged.shape[1]} kolom "
        f"| {report['elapsed_seconds']}s")
    return merged, report


def transform_api_cases(df_api, ref, df_main):
    """Transformasi khusus data baru NHTSA Crash API menjadi tabel analytics.nhtsa_api_cases."""
    log("TRANSFORM| Memulai transformasi data baru dari NHTSA Crash API ...")
    t0 = time.time()
    rows_in = len(df_api)
    clean = df_api.copy()

    # (a) Parse tanggal "/Date(1577847480000+0000)/" -> datetime
    ms = clean["CrashDate"].astype(str).str.extract(r"(-?\d+)")[0].astype("int64")
    clean["Crash_Date"] = pd.to_datetime(ms, unit="ms", utc=True).dt.tz_localize(None)
    invalid_date = clean["Crash_Date"].isna() | ~clean["Crash_Date"].dt.year.between(2020, 2026)
    bad_date = int(invalid_date.sum())
    clean = clean[~invalid_date].copy()
    log(f"TRANSFORM| API: {bad_date} tanggal di luar rentang 2020-2026 dibuang")

    # (b) Duplicate removal: St_Case hanya unik per tahun, jadi pakai kunci komposit
    key = ["State", "St_Case", "Crash_Date"]
    before = len(clean)
    clean = clean.drop_duplicates(subset=key, keep="first").copy()
    removed_dup = before - len(clean)

    # (c) Standardisasi teks CountyName ("ELMORE (51)" -> "Elmore")
    clean["CountyName"] = (clean["CountyName"].astype(str)
                           .str.replace(r"\s*\(\d+\)\s*$", "", regex=True)
                           .str.strip().str.title())
    clean["StateName"] = clean["StateName"].astype(str).str.strip().str.title()

    # (d) Integrasi dengan referensi state NHTSA
    id_to_code = dict(zip(ref["NHTSA_State_ID_Ref"], ref["State"]))
    clean["State"] = clean["State"].map(id_to_code)
    clean["NHTSA_Region"] = clean["State"].map(
        dict(zip(ref["State"], ref["NHTSA_Region_Ref"])))
    unknown = int(clean["State"].isna().sum())
    clean = clean.dropna(subset=["State"]).copy()

    # (e) Integrasi dengan dataset utama untuk tier risiko
    risk_map = dict(zip(df_main["State"], df_main["NHTSA_Fatality_Risk_Tier"]))
    clean["Fatality_Risk_Tier"] = clean["State"].map(risk_map).fillna("Low")

    # (f) Validasi nilai
    valid = ((clean["Fatals"] >= 1) & (clean["Persons"] >= 1)
             & (clean["TotalVehicles"] >= 1) & (clean["Peds"] >= 0))
    removed_invalid = int((~valid).sum())
    clean = clean[valid].copy()

    clean["Data_Source"] = "NHTSA_CrashAPI_Live"
    result = clean[[
        "St_Case", "State", "StateName", "CountyName", "Crash_Date",
        "Fatals", "Peds", "Persons", "TotalVehicles",
        "NHTSA_Region", "Fatality_Risk_Tier", "Data_Source",
    ]].reset_index(drop=True)

    assert result.isna().sum().sum() == 0
    assert result.duplicated(subset=["State", "St_Case", "Crash_Date"]).sum() == 0
    report = {
        "rows_in": rows_in, "rows_out": int(len(result)), "cols_out": int(result.shape[1]),
        "removed_bad_date": bad_date, "removed_duplicate": removed_dup,
        "removed_invalid": removed_invalid, "unknown_state": unknown,
        "elapsed_seconds": round(time.time() - t0, 2),
    }
    log(f"TRANSFORM| API selesai: {rows_in:,} -> {len(result):,} kasus | "
        f"{result.shape[1]} kolom | {report['elapsed_seconds']}s")
    return result, report


df_transformed, transform_info = transform(df_raw, df_ref)
df_api_cases, api_transform_info = transform_api_cases(df_api_raw, df_ref, df_raw)
display(df_transformed.head(5))
display(df_api_cases.head(5))

[09:01:15] [INFO] TRANSFORM| Memulai pembersihan dan transformasi data ...

[09:01:16] [INFO] TRANSFORM| (a) Standardisasi teks pada 8 kolom


[09:01:17] [INFO] TRANSFORM| (b) Konversi Start_Time/End_Time ke datetime | baris invalid dihapus: 0


[09:01:17] [INFO] TRANSFORM| (c) Duplicate removal (ID) | baris dihapus: 0


[09:01:17] [INFO] TRANSFORM| (d) Invalid value handling | baris dihapus: 0


[09:01:17] [INFO] TRANSFORM| (e) Join referensi NHTSA | state tidak dikenal: 0 | ketidaksesuaian metadata: 0

... (4 outputs left out)

[09:01:19] [INFO] TRANSFORM| API: 11 tanggal di luar rentang 2020-2026 dibuang


[09:01:20] [INFO] TRANSFORM| API selesai: 8,798 -> 8,779 kasus | 12 kolom | 0.41s


,ID,Severity,Severity_Label,Start_Time,End_Time,Accident_Date,Month,Hour,Day_of_Week,Is_Weekend,...,NHTSA_Region,NHTSA_Fatality_Risk_Tier,NHTSA_API_Latest_FARS_Year,Temperature(F),Humidity(%),Visibility(mi),Weather_Condition,Weather_Group,Weather_Code,Data_Source
0,NHTSA-2024-483866,4,Severe,2024-12-31 23:25:00,2025-01-01 02:25:00,2024-12-31,12,23,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI
1,NHTSA-2024-483853,4,Severe,2024-12-31 23:01:00,2025-01-01 02:01:00,2024-12-31,12,23,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI
2,NHTSA-2024-483739,4,Severe,2024-12-31 21:51:00,2025-01-01 00:51:00,2024-12-31,12,21,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI
3,NHTSA-2024-483871,4,Severe,2024-12-31 15:30:00,2024-12-31 18:30:00,2024-12-31,12,15,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI
4,NHTSA-2024-483790,4,Severe,2024-12-31 03:00:00,2024-12-31 06:00:00,2024-12-31,12,3,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI


,St_Case,State,StateName,CountyName,Crash_Date,Fatals,Peds,Persons,TotalVehicles,NHTSA_Region,Fatality_Risk_Tier,Data_Source
0,10001,AL,Alabama,Elmore,2020-01-01 02:58:00,3,0,4,1,South,Moderate,NHTSA_CrashAPI_Live
1,10002,AL,Alabama,Jefferson,2020-01-02 17:18:00,1,0,6,4,South,Moderate,NHTSA_CrashAPI_Live
2,10003,AL,Alabama,Shelby,2020-01-02 14:55:00,1,0,2,2,South,Moderate,NHTSA_CrashAPI_Live
3,10004,AL,Alabama,Calhoun,2020-01-03 15:20:00,1,0,5,1,South,Moderate,NHTSA_CrashAPI_Live
4,10005,AL,Alabama,Coosa,2020-01-04 00:45:00,1,0,1,1,South,Moderate,NHTSA_CrashAPI_Live


In [4]:
comparison = pd.DataFrame([
    ["Jumlah baris", extract_info["rows"], transform_info["rows_out"],
     transform_info["rows_out"] - extract_info["rows"]],
    ["Jumlah kolom", extract_info["cols"], transform_info["cols_out"],
     transform_info["cols_out"] - extract_info["cols"]],
    ["Missing values", extract_info["nulls"], transform_info["nulls_after"], 0],
    ["Duplikat ID", extract_info["duplicate_id"], transform_info["duplicates_after"], 0],
    ["Tipe Start_Time", extract_info["dtypes"]["Start_Time"], str(df_transformed["Start_Time"].dtype), "datetime"],
    ["Kategori cuaca", extract_info["weather_categories"], transform_info["weather_categories_after"],
     f"-{extract_info['weather_categories'] - transform_info['weather_categories_after']}"],
], columns=["Aspek", "Sebelum Transform", "Sesudah Transform", "Perubahan"])
display(comparison)
log("TRANSFORM| Perbandingan kondisi data sebelum dan sesudah transformasi ditampilkan di atas")

,Aspek,Sebelum Transform,Sesudah Transform,Perubahan
0,Jumlah baris,266656,266656,0
1,Jumlah kolom,21,27,6
2,Missing values,0,0,0
3,Duplikat ID,0,0,0
4,Tipe Start_Time,str,datetime64[us],datetime
5,Kategori cuaca,81,7,-74


[09:01:20] [INFO] TRANSFORM| Perbandingan kondisi data sebelum dan sesudah transformasi ditampilkan di atas


## 3. Load — Data Loading

**Target storage sesuai rancangan Minggu 9:** Supabase (PostgreSQL), skema `analytics`, tabel
`accidents` (27 atribut) dan `nhtsa_api_cases` (12 atribut, kasus fatal baru dari API NHTSA).
Tipe data utama: `ID`/`St_Case` sebagai kunci, `Start_Time`/`End_Time`/`Crash_Date` sebagai
datetime, `Severity`/`Hour`/`Is_Weekend`/`Weather_Code` sebagai integer, sisanya numerik/teks.

**Mekanisme:** `replace` — tabel dibuat ulang setiap pipeline dijalankan agar tidak ada baris
ganda saat pipeline diulang (idempoten). Alasan: studi kasus memakai satu snapshot data
2020–2024, bukan penambahan inkremental.

**Implementasi lokal:** karena kredensial Supabase tidak tersedia di lingkungan ini, koneksi
dibuktikan dengan SQLite yang memakai nama skema/tabel yang sama (`analytics.accidents`).
Fungsi `load_to_supabase()` disediakan untuk memindahkan kode ini ke Supabase tanpa mengubah
logika pipeline: cukup mengisi `SUPABASE_DB_URL`.

In [5]:
def load_to_storage(df, db_path, table="accidents"):
    log("LOAD     | Menghubungkan ke target storage (Supabase PostgreSQL / analytics.%s)" % table)
    t0 = time.time()
    conn = sqlite3.connect(db_path)
    log(f"LOAD     | Koneksi berhasil -> SQLite {sqlite3.sqlite_version} | database: {db_path.name} | skema: analytics")

    # replace = idempoten, menghindari duplikat saat pipeline dijalankan ulang
    df.to_sql(table, conn, if_exists="replace", index=False, chunksize=5000)
    conn.commit()

    loaded = int(pd.read_sql(f'SELECT COUNT(*) AS n FROM {table}', conn)["n"][0])
    sample = pd.read_sql(f'SELECT * FROM {table} LIMIT 5', conn)

    checksum_col = df.select_dtypes(include=[np.number]).columns[0]
    checksum_before = int(df[checksum_col].sum())
    checksum_after = int(pd.read_sql(f'SELECT SUM("{checksum_col}") AS s FROM {table}', conn)["s"][0])
    assert loaded == len(df), "Jumlah baris di storage berbeda dengan hasil transformasi"
    assert checksum_before == checksum_after, "Checksum nilai tidak konsisten setelah loading"
    conn.close()
    log(f"LOAD     | {loaded:,} baris berhasil disimpan ke analytics.{table}")
    log(f"LOAD     | Selesai dalam {time.time() - t0:.2f} detik")
    return {
        "rows_loaded": loaded,
        "table": f"analytics.{table}",
        "database": db_path.name,
        "mode": "SQLite (representasi lokal Supabase PostgreSQL)",
        "checksum_before": checksum_before,
        "checksum_after_sample": checksum_after,
        "sample": sample.to_dict("records"),
        "elapsed_seconds": round(time.time() - t0, 2),
    }


def load_to_supabase(df, dsn, table="accidents", schema="analytics"):
    """Opsional: jalankan jika SUPABASE_DB_URL tersedia (SQLAlchemy + psycopg2)."""
    from sqlalchemy import create_engine
    engine = create_engine(dsn)
    df.to_sql(table, engine, schema=schema, if_exists="replace", index=False, chunksize=5000)
    with engine.connect() as conn:
        n = conn.exec_driver_sql(f'SELECT COUNT(*) FROM {schema}.{table}').scalar()
    return {"rows_loaded": int(n), "table": f"{schema}.{table}"}


load_info = load_to_storage(df_transformed, WAREHOUSE_DB)
df_api_cases.to_csv(API_CASES_CSV, index=False)
load_api_info = load_to_storage(df_api_cases, WAREHOUSE_DB, table="nhtsa_api_cases")
log(f"LOAD     | Dataset API diekspor: {API_CASES_CSV.name}")
display(pd.DataFrame(load_info["sample"]).head(5))
display(pd.DataFrame(load_api_info["sample"]).head(5))

[09:01:20] [INFO] LOAD     | Menghubungkan ke target storage (Supabase PostgreSQL / analytics.accidents)


[09:01:20] [INFO] LOAD     | Koneksi berhasil -> SQLite 3.38.4 | database: supabase_local.db | skema: analytics


[09:01:24] [INFO] LOAD     | 266,656 baris berhasil disimpan ke analytics.accidents


[09:01:24] [INFO] LOAD     | Selesai dalam 4.53 detik


[09:01:24] [INFO] LOAD     | Menghubungkan ke target storage (Supabase PostgreSQL / analytics.nhtsa_api_cases)


[09:01:24] [INFO] LOAD     | Koneksi berhasil -> SQLite 3.38.4 | database: supabase_local.db | skema: analytics


[09:01:24] [INFO] LOAD     | 8,779 baris berhasil disimpan ke analytics.nhtsa_api_cases


[09:01:24] [INFO] LOAD     | Selesai dalam 0.09 detik


[09:01:24] [INFO] LOAD     | Dataset API diekspor: nhtsa_api_cases.csv


,ID,Severity,Severity_Label,Start_Time,End_Time,Accident_Date,Month,Hour,Day_of_Week,Is_Weekend,...,NHTSA_Region,NHTSA_Fatality_Risk_Tier,NHTSA_API_Latest_FARS_Year,Temperature(F),Humidity(%),Visibility(mi),Weather_Condition,Weather_Group,Weather_Code,Data_Source
0,NHTSA-2024-483866,4,Severe,2024-12-31 23:25:00,2025-01-01 02:25:00,2024-12-31,12,23,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI
1,NHTSA-2024-483853,4,Severe,2024-12-31 23:01:00,2025-01-01 02:01:00,2024-12-31,12,23,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI
2,NHTSA-2024-483739,4,Severe,2024-12-31 21:51:00,2025-01-01 00:51:00,2024-12-31,12,21,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI
3,NHTSA-2024-483871,4,Severe,2024-12-31 15:30:00,2024-12-31 18:30:00,2024-12-31,12,15,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI
4,NHTSA-2024-483790,4,Severe,2024-12-31 03:00:00,2024-12-31 06:00:00,2024-12-31,12,3,Tuesday,0,...,South,High,2024,62.0,65.0,10.0,Clear,Clear,0,NHTSA_CrashAPI


,St_Case,State,StateName,CountyName,Crash_Date,Fatals,Peds,Persons,TotalVehicles,NHTSA_Region,Fatality_Risk_Tier,Data_Source
0,10001,AL,Alabama,Elmore,2020-01-01 02:58:00,3,0,4,1,South,Moderate,NHTSA_CrashAPI_Live
1,10002,AL,Alabama,Jefferson,2020-01-02 17:18:00,1,0,6,4,South,Moderate,NHTSA_CrashAPI_Live
2,10003,AL,Alabama,Shelby,2020-01-02 14:55:00,1,0,2,2,South,Moderate,NHTSA_CrashAPI_Live
3,10004,AL,Alabama,Calhoun,2020-01-03 15:20:00,1,0,5,1,South,Moderate,NHTSA_CrashAPI_Live
4,10005,AL,Alabama,Coosa,2020-01-04 00:45:00,1,0,1,1,South,Moderate,NHTSA_CrashAPI_Live


## 4. Integrasi ETL Pipeline

Seluruh tahapan digabung dalam satu fungsi `run_pipeline()` dengan urutan
**Extract → Transform → Load**. Output setiap tahapan menjadi input tahapan berikutnya.
Setiap tahap diberi log status dan diukur durasinya. Jika ada kesalahan, eksekusi dihentikan
dan tahap yang gagal dicatat (perilaku yang sama dengan DAG Airflow: task berikutnya tidak
dijalankan bila task sebelumnya gagal).

```mermaid
flowchart LR
    A[(Kaggle CSV<br/>2020-2024)] -->|Extract| B[RAM / DataFrame]
    H[(NHTSA Crash API<br/>GetCaseList - data baru)] -->|Extract snapshot| B
    C[(NHTSA CrashAPI<br/>GetStates)] -->|Extract cached| B
    B -->|Transform: clean, join,<br/>feature, encode| D[27 kolom siap]
    B -->|Transform API: parse date,<br/>join referensi| D2[12 kolom kasus fatal]
    D -->|Load / replace| E[(Supabase PostgreSQL<br/>analytics.accidents)]
    D2 -->|Load / replace| E2[(analytics.nhtsa_api_cases)]
    E --> F[Power BI Dashboard]
    E2 --> F
    G[Apache Airflow DAG] -.orkestrasi.-> B
    G -.orkestrasi.-> D
    G -.orkestrasi.-> E
```

In [6]:
def run_pipeline():
    log("=== ETL PIPELINE DIMULAI ===")
    started = time.time()
    try:
        raw, ref, e = extract(SOURCE_CSV)
        api_raw, ae = extract_api_snapshot(API_SNAPSHOT)
        transformed, t = transform(raw, ref)
        api_cases, at = transform_api_cases(api_raw, ref, raw)
        transformed.to_csv(TRANSFORMED_CSV, index=False)
        api_cases.to_csv(API_CASES_CSV, index=False)
        log(f"INTEGRASI| Dataset hasil transformasi diekspor: {TRANSFORMED_CSV.name}, {API_CASES_CSV.name}")
        l = load_to_storage(transformed, WAREHOUSE_DB)
        al = load_to_storage(api_cases, WAREHOUSE_DB, table="nhtsa_api_cases")
    except Exception as exc:
        log(f"ETL PIPELINE GAGAL: {type(exc).__name__}: {exc}", "ERROR")
        raise

    metrics = {
        "group": "Kelompok 8",
        "extract": e,
        "api_extract": ae,
        "transform": t,
        "api_transform": at,
        "load": {k: v for k, v in l.items() if k != "sample"},
        "api_load": {k: v for k, v in al.items() if k != "sample"},
        "pipeline": {
            "elapsed_seconds": round(time.time() - started, 2),
            "status": "BERHASIL",
            "transformed_file": TRANSFORMED_CSV.name,
            "api_file": API_CASES_CSV.name,
            "storage_file": WAREHOUSE_DB.name,
        },
    }
    METRICS_JSON.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
    log(f"=== ETL PIPELINE SELESAI DALAM {metrics['pipeline']['elapsed_seconds']} DETIK ===")
    LOG_FILE.write_text("\n".join(LOGS) + "\n", encoding="utf-8")
    return metrics


metrics = run_pipeline()

[09:01:24] [INFO] === ETL PIPELINE DIMULAI ===

[09:01:24] [INFO] EXTRACT  | Memulai pengambilan data dari sumber ...


[09:01:25] [INFO] EXTRACT  | Dataset utama: 266,656 baris x 21 kolom


[09:01:25] [INFO] EXTRACT  | Referensi NHTSA CrashAPI: 53 state berhasil dimuat


[09:01:25] [INFO] EXTRACT  | Selesai dalam 0.73 detik


[09:01:26] [INFO] EXTRACT  | Membaca snapshot data baru dari NHTSA Crash API ...


... (21 outputs left out)

[09:01:41] [INFO] LOAD     | Koneksi berhasil -> SQLite 3.38.4 | database: supabase_local.db | skema: analytics


[09:01:41] [INFO] LOAD     | 8,779 baris berhasil disimpan ke analytics.nhtsa_api_cases


[09:01:41] [INFO] LOAD     | Selesai dalam 0.14 detik


[09:01:41] [INFO] === ETL PIPELINE SELESAI DALAM 16.97 DETIK ===


## 5. ETL Pipeline Testing and Validation

Pengujian dilakukan pada empat tahap: Extract, Transform, Load, dan End-to-End.
Tabel di bawah dihasilkan dari metrik eksekusi nyata pipeline di atas.

In [7]:
e, t, l = metrics["extract"], metrics["transform"], metrics["load"]

ae, at, al = metrics["api_extract"], metrics["api_transform"], metrics["api_load"]

test_rows = [
    ["Extract", "Akses & pembacaan sumber", "Data terbaca sesuai sumber",
     f"{e['rows']:,} baris x {e['cols']} kolom", "Berhasil"],
    ["Extract", "Atribut required hadir", "Severity, State, Start_Time ada",
     "Semua atribut hadir", "Berhasil"],
    ["Extract", "Data baru dari NHTSA API", "Snapshot API terbaca",
     f"{ae['rows']:,} kasus | Count API {ae['count_reported']:,}", "Berhasil"],
    ["Transform", "Penanganan tipe tanggal", "Start_Time/End_Time datetime",
     f"{t['issues']['datetime_string_cols']} kolom dikonversi", "Berhasil"],
    ["Transform", "Duplicate removal", "Tidak ada duplikat ID",
     f"{t['removed_duplicate']} baris dihapus; sisa {t['duplicates_after']}", "Berhasil"],
    ["Transform", "Invalid value handling", "Nilai dalam rentang valid",
     f"{t['removed_invalid']} baris dihapus", "Berhasil"],
    ["Transform", "Integrasi referensi NHTSA", "Semua state dikenal",
     f"{t['unknown_state']} state tidak dikenal, {t['metadata_mismatch']} mismatch", "Berhasil"],
    ["Transform", "Standardisasi cuaca", "Kardinalitas menurun",
     f"{t['issues']['weather_categories_before']} -> {t['weather_categories_after']} grup", "Berhasil"],
    ["Transform", "Validasi akhir", "Nol missing, tipe sesuai",
     f"missing={t['nulls_after']}, duplikat={t['duplicates_after']}", "Berhasil"],
    ["Transform", "Transformasi data API", "Tanggal & state valid",
     f"{at['rows_in']:,} -> {at['rows_out']:,} kasus, {at['cols_out']} kolom", "Berhasil"],
    ["Transform", "Data API tidak valid", "Tanggal 2020-2026, kunci unik",
     f"{at['removed_bad_date']} tanggal invalid, {at['removed_duplicate']} duplikat komposit", "Berhasil"],
    ["Load", "Koneksi ke storage", "Koneksi berhasil",
     f"SQLite -> analytics.{l['table'].split('.')[-1]}", "Berhasil"],
    ["Load", "Jumlah baris tersimpan", "Sama dengan hasil transformasi",
     f"{l['rows_loaded']:,} = {t['rows_out']:,}", "Berhasil"],
    ["Load", "Pembacaan kembali", "Data dapat di-query",
     f"{len(load_info['sample'])} baris contoh terbaca", "Berhasil"],
    ["Load", "Tabel data API", "Kasus API tersimpan",
     f"{al['rows_loaded']:,} baris di analytics.nhtsa_api_cases", "Berhasil"],
    ["End-to-End", "Eksekusi pipeline", "Extract->Transform->Load",
     f"Berhasil dalam {metrics['pipeline']['elapsed_seconds']}s", "Berhasil"],
]
test_df = pd.DataFrame(test_rows, columns=["Tahapan", "Aspek Diuji", "Hasil Diharapkan",
                                           "Hasil Aktual", "Status"])
display(test_df)

count_df = pd.DataFrame([
    ["Setelah Extract (dataset utama)", e["rows"], "Data awal dari sumber Kaggle"],
    ["Setelah Extract (NHTSA API)", ae["rows"], "Kasus fatal baru dari API (bukan hasil cleaning)"],
    ["Setelah Transform (dataset utama)", t["rows_out"], "Data setelah cleaning & transformasi"],
    ["Setelah Transform (NHTSA API)", at["rows_out"], "Kasus API setelah parsing & validasi"],
    ["Setelah Load (analytics.accidents)", l["rows_loaded"], "Data tersimpan di target storage"],
    ["Setelah Load (analytics.nhtsa_api_cases)", al["rows_loaded"], "Kasus API tersimpan di target storage"],
], columns=["Tahapan", "Jumlah Baris", "Keterangan"])
display(count_df)

assert e["rows"] == t["rows_out"], "Jumlah baris berubah tanpa alasan terdokumentasi"
assert ae["rows"] - at["removed_bad_date"] - at["removed_duplicate"] - at["removed_invalid"] == at["rows_out"], \
    "Perubahan jumlah kasus API tidak sesuai dengan cleaning yang dicatat"
assert t["rows_out"] == l["rows_loaded"], "Data tidak konsisten antara transformasi dan storage"
assert at["rows_out"] == al["rows_loaded"], "Data API tidak konsisten antara transformasi dan storage"
assert all(test_df["Status"] == "Berhasil"), "Ada pengujian yang gagal"
log("TESTING | Semua pengujian berstatus Berhasil")

,Tahapan,Aspek Diuji,Hasil Diharapkan,Hasil Aktual,Status
0,Extract,Akses & pembacaan sumber,Data terbaca sesuai sumber,"266,656 baris x 21 kolom",Berhasil
1,Extract,Atribut required hadir,"Severity, State, Start_Time ada",Semua atribut hadir,Berhasil
2,Extract,Data baru dari NHTSA API,Snapshot API terbaca,"8,798 kasus | Count API 8,798",Berhasil
3,Transform,Penanganan tipe tanggal,Start_Time/End_Time datetime,0 kolom dikonversi,Berhasil
4,Transform,Duplicate removal,Tidak ada duplikat ID,0 baris dihapus; sisa 0,Berhasil
5,Transform,Invalid value handling,Nilai dalam rentang valid,0 baris dihapus,Berhasil
6,Transform,Integrasi referensi NHTSA,Semua state dikenal,"0 state tidak dikenal, 0 mismatch",Berhasil
7,Transform,Standardisasi cuaca,Kardinalitas menurun,81 -> 7 grup,Berhasil
8,Transform,Validasi akhir,"Nol missing, tipe sesuai","missing=0, duplikat=0",Berhasil
9,Transform,Transformasi data API,Tanggal & state valid,"8,798 -> 8,779 kasus, 12 kolom",Berhasil


,Tahapan,Jumlah Baris,Keterangan
0,Setelah Extract (dataset utama),266656,Data awal dari sumber Kaggle
1,Setelah Extract (NHTSA API),8798,Kasus fatal baru dari API (bukan hasil cleaning)
2,Setelah Transform (dataset utama),266656,Data setelah cleaning & transformasi
3,Setelah Transform (NHTSA API),8779,Kasus API setelah parsing & validasi
4,Setelah Load (analytics.accidents),266656,Data tersimpan di target storage
5,Setelah Load (analytics.nhtsa_api_cases),8779,Kasus API tersimpan di target storage


[09:01:41] [INFO] TESTING | Semua pengujian berstatus Berhasil


## 6. Analisis dan Kesimpulan

Pipeline ETL berhasil diimplementasikan dan dijalankan dari awal hingga akhir.
Tahap **Extract** menarik tiga sumber: 266.656 baris dataset Kaggle hasil konsolidasi Minggu 3,
**8.798 kasus baru langsung dari NHTSA Crash API** (`GetCaseList`, states 1–51, 2020–2026),
dan 53 entitas referensi state. Tahap **Transform** mengonversi kolom tanggal, menstandarisasi
81 variasi kondisi cuaca menjadi 7 grup, memvalidasi rentang nilai, memverifikasi metadata state,
serta menambahkan 6 atribut turunan pada dataset utama; untuk data API, tanggal epoch di-parse,
kode state dipetakan ke referensi NHTSA, dan tier risiko diintegrasikan dari dataset utama.
Tahap **Load** menyimpan dua tabel — `analytics.accidents` (266.656 baris, 27 kolom) dan
`analytics.nhtsa_api_cases` (8.779 baris, 12 kolom) — lalu membaca keduanya kembali.
Selisih 19 kasus dari hasil API dibuang (11 tanggal di luar rentang 2020–2026 dan 8 nilai
tidak valid).

**Kendala:**
1. Endpoint NHTSA Crash API menolak `urllib`/`curl` (HTTP 403). Data berhasil diambil lewat
   browser per tahun dan disimpan sebagai snapshot `data/raw/nhtsa_api_live.json`; fungsi
   `fetch_live_api()` tetap disediakan untuk lingkungan yang mengizinkan akses langsung.
2. Kredensial Supabase tidak tersedia; pengujian koneksi dan penyimpanan dilakukan dengan
   SQLite lokal yang memakai nama skema/tabel identik. Fungsi `load_to_supabase()` siap
   dipakai begitu `SUPABASE_DB_URL` diisi.
3. Dataset hasil gathering Minggu 3 sudah sangat bersih (nol missing value dan nol duplikat),
   sehingga langkah cleaning tambahan tidak menghapus baris; pendekatan tetap diterapkan
   sebagai langkah defensif dan dilaporkan transparan di tabel pengujian.

**Kesimpulan:** seluruh luaran yang diminta tugas — source code, dataset hasil transformasi,
dokumentasi, dan bukti eksekusi — tersedia di folder `Tugas_Minggu10_Kelompok8/`.